# 05 -- Top 100 S&P 500 Universe Construction

## Purpose
Identifies the top 100 S&P 500 stocks by market capitalisation for each year from 2004 to 2024. The output defines the equity universe for the entire project -- all subsequent data collection, cleaning, and modelling is restricted to stocks that appear in this universe.

## Source
WRDS via the `wrds` Python library, authenticated with username `henrylavender`. Two tables are queried:
- `crsp_a_indexes.dsp500list` -- S&P 500 constituent history (join date, leave date for every stock that has ever been in the index)
- `crsp_a_stock.dsf_v2` -- CRSP daily stock file v2 (used for market cap and ticker lookups)

## Method

### Step 1: S&P 500 Membership
The full constituent history is pulled from `crsp_a_indexes.dsp500list`. For each year Y (2004--2024), a stock is classified as an S&P 500 member if it joined on or before January 1 of year Y and either has not left the index or left on or after January 1.

### Step 2: Market Cap Ranking
For each year Y, market capitalisation (`dlycap`) is retrieved from `crsp_a_stock.dsf_v2` for the last trading day of December Y-1 (i.e., 2004 rankings use December 2003 market caps). The query is filtered to common equity only:
- `sharetype = 'NS'` (normal shares)
- `securitytype = 'EQTY'` (equity)
- `primaryexch IN ('N', 'A', 'Q')` (NYSE, AMEX, NASDAQ)

The most recent non-null `dlycap` in December of the prior year is kept for each PERMNO. If no December data is found, a fallback query extends the window back to October. Queries are executed one year at a time to avoid pulling the full daily stock file.

### Step 3: Selection
All S&P 500 members for each year are ranked by descending `dlycap`. The top 100 are retained per year.

## Verification
- Top 5 stocks are printed for sample years (2004, 2010, 2015, 2020, 2024) with ticker symbols to visually confirm the ranking is sensible
- Year-over-year turnover is computed (stocks entering/exiting the top 100 each year); expected range is ~5--15 per year for large-cap stability
- Master list summary statistics: count of PERMNOs appearing in all 21 years, only 1 year, median/mean appearances

## Outputs
- `Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_annual.parquet` -- 100 PERMNOs per year with columns: `permno`, `year`, `dlycap`, `rank`
- `Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet` -- all unique PERMNOs that appear in any year's top 100 (single column: `permno`)

In [ ]:
# %% [markdown]
# # Stage 1: Build the Annual Top-100 S&P 500 Universe
#
# This notebook identifies the top 100 S&P 500 stocks by market capitalisation
# for each year from 2004 to 2024. The output is two files:
# - `universe_annual.parquet` — 100 PERMNOs per year with their market cap and rank
# - `universe_master.parquet` — all unique PERMNOs that ever appeared in any year's top 100
#
# These files gate all subsequent data downloads — we only collect data for stocks in this universe.

# %% [markdown]
# ## Setup

# %%
import wrds
import pandas as pd
import numpy as np
from pathlib import Path



conn = wrds.Connection(wrds_username='henrylavender')

START_YEAR = 2004
END_YEAR = 2024

# %% [markdown]
# ## Step 1: Pull the full S&P 500 constituent history
#
# The table `crsp_a_indexes.dsp500list` contains every stock that has ever been
# in the S&P 500, with the date it joined (`start`) and the date it left (`ending`).
# Stocks still in the index have `ending = NULL`.

# %%
sp500 = conn.raw_sql("""
    SELECT permno, start, ending
    FROM crsp_a_indexes.dsp500list
""", date_cols=['start', 'ending'])

print(f"S&P 500 constituent history: {len(sp500)} rows, {sp500['permno'].nunique()} unique PERMNOs")
print(f"Date range: {sp500['start'].min()} to {sp500['ending'].max()}")
print(f"Stocks currently in index (ending is NaT): {sp500['ending'].isna().sum()}")
sp500.head()

# %% [markdown]
# ## Step 2: For each year, determine S&P 500 membership as of January 1
#
# A stock is a member for year Y if it joined on or before Jan 1 of that year,
# and either hasn't left yet (ending is NULL) or left on or after Jan 1.
# This gives us the ~500 constituents for each year.

# %%
membership_rows = []

for year in range(START_YEAR, END_YEAR + 1):
    ref_date = pd.Timestamp(f'{year}-01-01')
    
    members = sp500[
        (sp500['start'] <= ref_date) &
        ((sp500['ending'] >= ref_date) | (sp500['ending'].isna()))
    ]['permno'].unique()
    
    for p in members:
        membership_rows.append({'permno': int(p), 'year': year})

membership = pd.DataFrame(membership_rows)
print(f"Total membership rows: {len(membership)}")
print(f"Members per year:")
print(membership.groupby('year')['permno'].nunique().to_string())

# %% [markdown]
# ## Step 3: Get market cap for each member to rank them
#
# For each year Y, we pull the market cap (`dlycap`) from `crsp_a_stock.dsf_v2`
# as of the last trading day of December Y-1. We filter to common equity only:
# - `sharetype = 'NS'` (normal shares)
# - `securitytype = 'EQTY'` (equity)
# - `primaryexch IN ('N', 'A', 'Q')` (NYSE, AMEX, NASDAQ)
#
# We take the most recent non-null `dlycap` in December of the prior year for each PERMNO.
# For the first year (2004), this means December 2003 market caps.
#
# We query one year at a time to avoid pulling the entire 110M-row table.

# %%
all_caps = []

for year in range(START_YEAR, END_YEAR + 1):
    # PERMNOs that are S&P 500 members for this year
    year_permnos = membership[membership['year'] == year]['permno'].tolist()
    
    if not year_permnos:
        continue
    
    permno_str = ','.join(str(p) for p in year_permnos)
    
    # Look up December of the prior year
    dec_start = f'{year - 1}-12-01'
    dec_end = f'{year - 1}-12-31'
    
    # Pull the last non-null dlycap in December Y-1 for each PERMNO
    query = f"""
        SELECT permno, dlycaldt, dlycap
        FROM crsp_a_stock.dsf_v2
        WHERE permno IN ({permno_str})
          AND dlycaldt BETWEEN '{dec_start}' AND '{dec_end}'
          AND sharetype = 'NS'
          AND securitytype = 'EQTY'
          AND primaryexch IN ('N', 'A', 'Q')
          AND dlycap IS NOT NULL
        ORDER BY permno, dlycaldt DESC
    """
    
    df = conn.raw_sql(query, date_cols=['dlycaldt'])
    
    if df.empty:
        print(f"  WARNING: No December {year-1} data found. Trying broader fallback.")
        # Fallback: last 3 months before Jan 1
        query_fb = f"""
            SELECT permno, dlycaldt, dlycap
            FROM crsp_a_stock.dsf_v2
            WHERE permno IN ({permno_str})
              AND dlycaldt BETWEEN '{year - 1}-10-01' AND '{dec_end}'
              AND sharetype = 'NS'
              AND securitytype = 'EQTY'
              AND primaryexch IN ('N', 'A', 'Q')
              AND dlycap IS NOT NULL
            ORDER BY permno, dlycaldt DESC
        """
        df = conn.raw_sql(query_fb, date_cols=['dlycaldt'])
    
    # Keep only the most recent observation per PERMNO (latest dlycaldt)
    caps = df.sort_values('dlycaldt', ascending=False).drop_duplicates(subset='permno', keep='first')
    caps = caps[['permno', 'dlycap']].copy()
    caps['year'] = year
    
    all_caps.append(caps)
    
    print(f"  Year {year}: {len(caps)} PERMNOs with market cap data "
          f"(out of {len(year_permnos)} members)")

mktcap = pd.concat(all_caps, ignore_index=True)
print(f"\nTotal market cap records: {len(mktcap)}")

# %% [markdown]
# ## Step 4: Rank and select top 100 per year
#
# For each year, we rank all S&P 500 members by descending `dlycap` and keep the top 100.

# %%
mktcap['rank'] = mktcap.groupby('year')['dlycap'].rank(ascending=False, method='first').astype(int)

universe_annual = (
    mktcap[mktcap['rank'] <= 100]
    .sort_values(['year', 'rank'])
    .reset_index(drop=True)
)

# Reorder columns
universe_annual = universe_annual[['permno', 'year', 'dlycap', 'rank']]

print(f"Universe annual shape: {universe_annual.shape}")
print(f"Rows per year (should all be 100):")
print(universe_annual.groupby('year').size().to_string())

# %% [markdown]
# ## Step 5: Save outputs
#
# 1. `universe_annual.parquet` — the full annual top-100 list (2,100 rows)
# 2. `universe_master.parquet` — all unique PERMNOs that appear in any year (the "master list")

# %%
# Save annual universe
universe_annual.to_parquet('../../Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_annual.parquet', index=False, engine='pyarrow')
print(f"Saved universe_annual.parquet: {universe_annual.shape}")

# Build and save master list
master_permnos = pd.DataFrame({
    'permno': universe_annual['permno'].unique()
})
master_permnos.to_parquet('../../Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet', index=False, engine='pyarrow')
print(f"Saved universe_master.parquet: {len(master_permnos)} unique PERMNOs")

# %% [markdown]
# ## Step 6: Verification & Summary Stats

# %% [markdown]
# ### 6a: Top 5 stocks in sample years
#
# Checking that the largest PERMNOs in key years are recognisable names (Apple, Microsoft, etc.)
# We pull ticker symbols for the top 5 in each sample year so we can verify visually.

# %%
# Get ticker info for our universe PERMNOs
permno_str = ','.join(str(p) for p in master_permnos['permno'].tolist())
tickers = conn.raw_sql(f"""
    SELECT DISTINCT permno, ticker
    FROM crsp_a_stock.dsf_v2
    WHERE permno IN ({permno_str})
      AND dlycaldt >= '2024-12-01'
      AND ticker IS NOT NULL
""")
# Keep one ticker per permno (the most recent)
ticker_map = tickers.drop_duplicates(subset='permno', keep='last').set_index('permno')['ticker']

# Also get historical tickers for older years
tickers_hist = conn.raw_sql(f"""
    SELECT DISTINCT permno, ticker, dlycaldt
    FROM crsp_a_stock.dsf_v2
    WHERE permno IN ({permno_str})
      AND ticker IS NOT NULL
    ORDER BY permno, dlycaldt DESC
""", date_cols=['dlycaldt'])
ticker_hist_map = tickers_hist.drop_duplicates(subset='permno', keep='first').set_index('permno')['ticker']

# Merge both — prefer recent, fall back to historical
full_ticker_map = ticker_hist_map.copy()
full_ticker_map.update(ticker_map)

for sample_year in [2004, 2010, 2015, 2020, 2024]:
    top5 = universe_annual[universe_annual['year'] == sample_year].head(5).copy()
    top5['ticker'] = top5['permno'].map(full_ticker_map)
    top5['dlycap_bn'] = (top5['dlycap'] / 1e6).round(1)  # dlycap is in thousands → /1e6 = billions
    print(f"\n--- Top 5 in {sample_year} ---")
    print(top5[['rank', 'permno', 'ticker', 'dlycap_bn']].to_string(index=False))

# %% [markdown]
# ### 6b: Year-over-year turnover
#
# How many stocks enter/exit the top 100 each year? Low turnover (~5-15) indicates
# the universe is stable, which is what we expect for large-cap S&P 500 stocks.

# %%
print("\nYear-over-year turnover (stocks entering the top 100):")
prev_set = None
for year in range(START_YEAR, END_YEAR + 1):
    curr_set = set(universe_annual[universe_annual['year'] == year]['permno'])
    if prev_set is not None:
        entered = len(curr_set - prev_set)
        exited = len(prev_set - curr_set)
        print(f"  {year}: +{entered} entered, -{exited} exited")
    prev_set = curr_set

# %% [markdown]
# ### 6c: Master list summary

# %%
n_master = len(master_permnos)
n_years = END_YEAR - START_YEAR + 1

# How many years does each PERMNO appear in?
appearances = universe_annual.groupby('permno')['year'].nunique()

print(f"\nMaster list: {n_master} unique PERMNOs across {n_years} years")
print(f"PERMNOs in top 100 for ALL {n_years} years: {(appearances == n_years).sum()}")
print(f"PERMNOs in top 100 for only 1 year: {(appearances == 1).sum()}")
print(f"Median years in top 100: {appearances.median():.0f}")
print(f"Mean years in top 100: {appearances.mean():.1f}")

# %% [markdown]
# ## Cleanup

# %%
conn.close()
print("WRDS connection closed.")
print("\nStage 1 complete. Files saved:")
print("  data/universe_annual.parquet")
print("  data/universe_master.parquet")